In [30]:
import io, requests, pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
# EDUCATION ATTAINMENT DATASET



BASE   = "https://sdmx.oecd.org/public/rest"
AGENCY = "OECD.CFE.EDS"
FLOW   = "DSD_REG_EDU@DF_ATTAIN"
VER    = "2.0"

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
    "Accept": "application/json,text/csv,*/*;q=0.8",
}

# 1) Get dimension structure
probe = requests.get(
    f"{BASE}/data/{AGENCY},{FLOW},{VER}",
    params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
    headers=headers, timeout=60
)
probe.raise_for_status()
js = probe.json()

series_dims = js["structure"]["dimensions"]["series"]
dim_order = [d["id"] for d in series_dims]
print("Education dataset dimensions:", dim_order)

# Build codes lookup
codes = {
    d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))}
              for v in d.get("values", [])]
    for d in series_dims
}

def pick_code(dim, prefer_ids=(), prefer_name_contains=()):
    """Pick a valid code for a dimension using id or name hints."""
    vals = codes.get(dim, [])
    # Try id preferences
    for pid in prefer_ids:
        for v in vals:
            if v["id"].upper() == pid.upper():
                return v["id"]
    # Try name contains
    for substr in prefer_name_contains:
        for v in vals:
            if substr.lower() in str(v["name"]).lower():
                return v["id"]
    # Fallback: first available (if any)
    return vals[0]["id"] if vals else ""

# 2) Build proper key with ALL 10 dimensions for education dataset
freq = pick_code("FREQ", prefer_ids=("A",))                                    # Annual
terr_level = "TL2+CTRY"                                                       # BOTH TL2 regions AND countries
ref_area = ""                                                                 # All regions
terr_type = pick_code("TERRITORIAL_TYPE", prefer_ids=("_Z",))                 # Not applicable
measure = pick_code("MEASURE", prefer_ids=("NEAC_SHARE_EA",))                 # Education attainment
age = pick_code("AGE", prefer_ids=("Y25T64",))                                # 25-64 years
sex = pick_code("SEX", prefer_ids=("_T", "T"), prefer_name_contains=("total",)) # Total
education_lev = pick_code("EDUCATION_LEV", prefer_ids=("ISCED11_5T8",))       # Tertiary education
stat_op = pick_code("STATISTICAL_OPERATION", prefer_ids=("MEAN",))            # Mean
unit = pick_code("UNIT_MEASURE", prefer_ids=("PT_POP_SEX_AGE",))              # Percentage

# Build complete key with all 10 dimensions
key_parts = [freq, terr_level, ref_area, terr_type, measure, age, sex, education_lev, stat_op, unit]
key = ".".join(key_parts)
print(f"Education key ({len(key_parts)} parts):", key)

# 3) Download data
url = f"{BASE}/data/{AGENCY},{FLOW},{VER}/{key}"
params = {
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels",
    "startPeriod": "2010",
    "endPeriod": "2024",
}

r = requests.get(url, params=params, headers=headers, timeout=60)

try:
    r.raise_for_status()
    education_df = pd.read_csv(io.StringIO(r.text))
    print(f"\n Education data loaded: {education_df.shape}")
    print("\nFirst few rows:")
    print(education_df.head())
    
    # Show what we got
    print("\nDataset contains:")
    for col in education_df.columns:
        if any(keyword in col.lower() for keyword in ['measure', 'education', 'age', 'territorial']):
            unique_vals = education_df[col].dropna().unique()
            print(f"  {col}: {unique_vals[:3]}{'...' if len(unique_vals) > 3 else ''}")
            
except requests.HTTPError as e:
    print(f" Still failed with: {e}")
    print("Available dimension codes:")
    for d in dim_order:
        print(f"  {d}: {[v['id'] for v in codes.get(d, [])][:5]}")

Education dataset dimensions: ['FREQ', 'TERRITORIAL_LEVEL', 'REF_AREA', 'TERRITORIAL_TYPE', 'MEASURE', 'AGE', 'SEX', 'EDUCATION_LEV', 'STATISTICAL_OPERATION', 'UNIT_MEASURE']
Education key (10 parts): A.TL2+CTRY.._Z.NEAC_SHARE_EA.Y25T64._T.ISCED11_5T8.MEAN.PT_POP_SEX_AGE

 Education data loaded: (6771, 36)

First few rows:
  STRUCTURE                             STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   

                     STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Educational attainment - Regions      I    A                   Annual   
1  Educational attainment - Regions      I    A                   Annual   
2  Educational attainment - Regions      I    A                   Annual   
3  Educational attainment - Region

In [7]:
# Check the data structure and identify time-related columns
print("All columns in the dataset:")
print(education_df.columns.tolist())

print("\nLooking for time/date related columns:")
time_cols = [col for col in education_df.columns if any(keyword in col.lower() 
            for keyword in ['time', 'date', 'year', 'period', 'obs_time'])]
print("Time-related columns:", time_cols)

# Check the first few values of potential time columns
for col in time_cols:
    print(f"\n{col} sample values:")
    print(education_df[col].dropna().unique()[:15])

print(f"\nDataset shape: {education_df.shape}")

All columns in the dataset:
['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'TERRITORIAL_TYPE', 'Territorial typology', 'MEASURE', 'Measure', 'AGE', 'Age', 'SEX', 'Sex', 'EDUCATION_LEV', 'Education level', 'STATISTICAL_OPERATION', 'Statistical operation', 'UNIT_MEASURE', 'Unit of measure', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country', 'OBS_STATUS', 'Observation status', 'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals']

Looking for time/date related columns:
Time-related columns: ['TIME_PERIOD', 'Time period']

TIME_PERIOD sample values:
[2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2021 2022 2020 2023
 2024]

Time period sample values:
[]

Dataset shape: (6771, 36)


In [153]:
# CHECK WHAT COUNTRIES HAVE DATA FOR 2023
print("COUNTRIES WITH 2022 DATA")
print("=" * 50)

# First, check if 2023 data exists at all
data_2023 = education_df[education_df['TIME_PERIOD'] == 2022]
print(f" Total 2023 observations: {len(data_2023)}")

if len(data_2023) > 0:
    # Get unique countries for 2023
    countries_2023 = data_2023['Country'].unique()
    print(f"\n Countries with 2023 data ({len(countries_2023)}):")
    for i, country in enumerate(sorted(countries_2023), 1):
        print(f"  {i:2d}. {country}")
    
    # Show some sample regions for 2023
    print(f"\n Sample regions with 2023 data:")
    sample_regions = data_2023[['Country', 'Reference area', 'OBS_VALUE']].head(10)
    print(sample_regions.to_string(index=False))
    
    # Value statistics for 2023
    values_2023 = data_2023['OBS_VALUE'].dropna()
    if len(values_2023) > 0:
        print(f"\n 2023 Education Attainment Statistics:")
        print(f"  • Number of regions: {len(values_2023)}")
        print(f"  • Range: {values_2023.min():.1f}% - {values_2023.max():.1f}%")
        print(f"  • Average: {values_2023.mean():.1f}%")
        print(f"  • Median: {values_2023.median():.1f}%")
        
else:
    print("\n No 2023 data found in the dataset")
    print("\nAvailable years:")
    available_years = sorted(education_df['TIME_PERIOD'].unique())
    print(f"  {available_years}")
    print(f"\nLatest year available: {max(available_years)}")
    
    # Show countries for the latest available year instead
    latest_year = max(available_years)
    latest_data = education_df[education_df['TIME_PERIOD'] == latest_year]
    countries_latest = latest_data['Country'].unique()
    print(f"\n🌍 Countries with {latest_year} data ({len(countries_latest)}):")
    for i, country in enumerate(sorted(countries_latest), 1):
        print(f"  {i:2d}. {country}")

COUNTRIES WITH 2022 DATA
 Total 2023 observations: 444

 Countries with 2023 data (42):
   1. Austria
   2. Belgium
   3. Bulgaria
   4. Canada
   5. Chile
   6. Colombia
   7. Costa Rica
   8. Croatia
   9. Cyprus
  10. Czechia
  11. Denmark
  12. Estonia
  13. Euro area (20 countries)
  14. European Union (27 countries from 01/02/2020)
  15. Finland
  16. France
  17. Germany
  18. Greece
  19. Hungary
  20. Iceland
  21. Ireland
  22. Israel
  23. Italy
  24. Latvia
  25. Lithuania
  26. Luxembourg
  27. Malta
  28. Mexico
  29. Netherlands
  30. Norway
  31. Poland
  32. Portugal
  33. Romania
  34. Serbia
  35. Slovak Republic
  36. Slovenia
  37. Spain
  38. Sweden
  39. Switzerland
  40. Türkiye
  41. United Kingdom
  42. United States

 Sample regions with 2023 data:
      Country Reference area  OBS_VALUE
United States       Virginia       52.8
United States         Nevada       35.9
United States       Michigan       44.7
United States   North Dakota       50.3
United States 

In [9]:
# lets work with 2022 data
data_2022 = education_df[education_df['TIME_PERIOD'] == 2022]
data_2022_us = data_2022[data_2022['COUNTRY'] == 'USA']
vars = data_2022[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'Education level', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
vars

,STRUCTURE_NAME,TERRITORIAL_LEVEL,Territorial level,REF_AREA,Reference area,MEASURE,Measure,Age,Sex,Education level,TIME_PERIOD,Time period,OBS_VALUE,Observation value,COUNTRY,Country
557,Educational attainment - Regions,TL2,TL2,US51,Virginia,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,52.8,NaN,USA,United States
558,Educational attainment - Regions,TL2,TL2,US32,Nevada,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,35.9,NaN,USA,United States
559,Educational attainment - Regions,TL2,TL2,US26,Michigan,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,44.7,NaN,USA,United States
560,Educational attainment - Regions,TL2,TL2,US38,North Dakota,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,50.3,NaN,USA,United States
561,Educational attainment - Regions,TL2,TL2,US41,Oregon,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,47.1,NaN,USA,United States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5931,Educational attainment - Regions,TL2,TL2,ITF1,Abruzzo,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,21.6,NaN,ITA,Italy
5932,Educational attainment - Regions,TL2,TL2,ITH4,Friuli-Venezia Giulia,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,21.3,NaN,ITA,Italy
5933,Educational attainment - Regions,TL2,TL2,ITH1,Autonomous prov. Bolzano,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,18.1,NaN,ITA,Italy
5934,Educational attainment - Regions,TL2,TL2,ITC2,Aosta Valley,NEAC_SHARE_EA,Population educational attainment,From 25 to 64 years,Total,Tertiary education,2022,NaN,19.6,NaN,ITA,Italy


In [ ]:
data_2022 = education_df[education_df['TIME_PERIOD'] == 2022]
data_2022_us = data_2022[data_2022['COUNTRY'] == 'USA']
vars = data_2022[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'Education level', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]

# Separate the two territorial levels
EDU_country_data = vars[vars['Territorial level'] == 'Country']
EDU_regional_data = vars[vars['Territorial level'] == 'TL2']


In [168]:
# CALCULATE USA AND UK COUNTRY VALUES AS MEAN OF REGIONAL VALUES
countries_to_add = []

# Get USA regional data
usa_regional = EDU_regional_data[EDU_regional_data['COUNTRY'] == 'USA']
if len(usa_regional) > 0:
    usa_mean = usa_regional['OBS_VALUE'].mean()
    usa_country_row = {
        'STRUCTURE_NAME': usa_regional.iloc[0]['STRUCTURE_NAME'],
        'TERRITORIAL_LEVEL': 'CTRY',
        'Territorial level': 'Country',
        'REF_AREA': 'USA',
        'Reference area': 'United States',
        'MEASURE': usa_regional.iloc[0]['MEASURE'],
        'Measure': usa_regional.iloc[0]['Measure'],
        'Age': usa_regional.iloc[0]['Age'],
        'Sex': usa_regional.iloc[0]['Sex'],
        'Education level': usa_regional.iloc[0]['Education level'],
        'TIME_PERIOD': 2022,
        'Time period': '2022',
        'OBS_VALUE': usa_mean,
        'Observation value': usa_mean,
        'COUNTRY': 'USA',
        'Country': 'United States'
    }
    countries_to_add.append(usa_country_row)
    print(f" USA mean calculated: {usa_mean:.2f}% from {len(usa_regional)} regions")
else:
    print(" No USA regional data found")

# Get UK regional data  
uk_regional = EDU_regional_data[EDU_regional_data['COUNTRY'] == 'GBR']
if len(uk_regional) > 0:
    uk_mean = uk_regional['OBS_VALUE'].mean()
    uk_country_row = {
        'STRUCTURE_NAME': uk_regional.iloc[0]['STRUCTURE_NAME'],
        'TERRITORIAL_LEVEL': 'CTRY',
        'Territorial level': 'Country',
        'REF_AREA': 'GBR',
        'Reference area': 'United Kingdom',
        'MEASURE': uk_regional.iloc[0]['MEASURE'],
        'Measure': uk_regional.iloc[0]['Measure'],
        'Age': uk_regional.iloc[0]['Age'],
        'Sex': uk_regional.iloc[0]['Sex'],
        'Education level': uk_regional.iloc[0]['Education level'],
        'TIME_PERIOD': 2022,
        'Time period': '2022',
        'OBS_VALUE': uk_mean,
        'Observation value': uk_mean,
        'COUNTRY': 'GBR',
        'Country': 'United Kingdom'
    }
    countries_to_add.append(uk_country_row)
 
else:
    print("No UK regional data found")

# Add both countries to country data if we have any
if countries_to_add:
    EDU_country_data_with_additions = pd.concat([
        EDU_country_data, 
        pd.DataFrame(countries_to_add)
    ], ignore_index=True)
    
    countries_with_additions = EDU_country_data_with_additions[['Reference area', 'OBS_VALUE']].sort_values('OBS_VALUE', ascending=False)
    
    print(countries_with_additions.head(10))
    
else:
    print(" No countries could be added")
    EDU_country_data_with_additions = EDU_country_data
    countries_with_additions = EDU_country_data[['Reference area', 'OBS_VALUE']].sort_values('OBS_VALUE', ascending=False)

 USA mean calculated: 47.01% from 51 regions
    Reference area  OBS_VALUE
26          Canada  63.000000
23         Ireland  53.700000
25      Luxembourg  52.300000
33          Israel  50.300000
41  United Kingdom  48.675000
37          Sweden  48.500000
4           Cyprus  48.000000
18          Norway  47.800000
40   United States  47.011765
17       Lithuania  46.500000


In [146]:
countries_with_additions.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_countries_complete.csv", index=False)
EDU_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_regional_data.csv", index=False)

# Getting Health Data

In [ ]:
# Life expectancy

BASE   = "https://sdmx.oecd.org/public/rest"
AGENCY = "OECD.CFE.EDS"
FLOW   = "DSD_REG_HEALTH@DF_HEALTH"
VER    = "2.0"

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
    "Accept": "application/json,text/csv,*/*;q=0.8",
}

# 1) Get dimension structure
probe = requests.get(
    f"{BASE}/data/{AGENCY},{FLOW},{VER}",
    params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
    headers=headers, timeout=60
)
probe.raise_for_status()
js = probe.json()

series_dims = js["structure"]["dimensions"]["series"]
dim_order = [d["id"] for d in series_dims]
print("life expectancy dataset dimensions:", dim_order)

# Build codes lookup
codes = {
    d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))}
              for v in d.get("values", [])]
    for d in series_dims
}

def pick_code(dim, prefer_ids=(), prefer_name_contains=()):
    """Pick a valid code for a dimension using id or name hints."""
    vals = codes.get(dim, [])
    # Try id preferences
    for pid in prefer_ids:
        for v in vals:
            if v["id"].upper() == pid.upper():
                return v["id"]
    # Try name contains
    for substr in prefer_name_contains:
        for v in vals:
            if substr.lower() in str(v["name"]).lower():
                return v["id"]
    # Fallback: first available (if any)
    return vals[0]["id"] if vals else ""

# 2) Build proper key with ALL 10 dimensions for education dataset
# Build proper key for health dataset
freq = pick_code("FREQ", prefer_ids=("A",))
terr_level = "TL2+CTRY"
ref_area = ""  # All regions/countries, or specify as in SDMX code
terr_type = "" # Usually empty for health
measure = pick_code("MEASURE", prefer_ids=("LFEXP",))  # Life expectancy
age = pick_code("AGE", prefer_ids=("Y0",))              # All ages
sex = ""  # Or "F+M+_T" if you want all sexes
unit = "" # Usually empty for health

# Build complete key (omit education_lev and stat_op)
key_parts = [freq, terr_level, ref_area, terr_type, measure, age, sex, unit]
key = ".".join(key_parts)
print(f"Health key ({len(key_parts)} parts):", key)

# 3) Download data
url = f"{BASE}/data/{AGENCY},{FLOW},{VER}/{key}"
params = {
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels",
    "startPeriod": "2010",
    "endPeriod": "2024",
}

r = requests.get(url, params=params, headers=headers, timeout=60)

try:
    r.raise_for_status()
    health_df = pd.read_csv(io.StringIO(r.text))
    print(f"\n Health data loaded: {health_df.shape}")
    print("\nFirst few rows:")
    print(health_df.head())

    # Show what we got
    print("\nDataset contains:")
    for col in health_df.columns:
        if any(keyword in col.lower() for keyword in ['measure', 'health', 'age', 'territorial']):
            unique_vals = health_df[col].dropna().unique()
            print(f"  {col}: {unique_vals[:3]}{'...' if len(unique_vals) > 3 else ''}")
except requests.HTTPError as e:
    print(f" Still failed with: {e}")
    print("Available dimension codes:")
    for d in dim_order:
        print(f"  {d}: {[v['id'] for v in codes.get(d, [])][:5]}")
# ...existing code...

life expectancy dataset dimensions: ['FREQ', 'TERRITORIAL_LEVEL', 'REF_AREA', 'TERRITORIAL_TYPE', 'MEASURE', 'AGE', 'SEX', 'UNIT_MEASURE']
Health key (8 parts): A.TL2+CTRY...LFEXP.Y0..

 Health data loaded: (20022, 32)

First few rows:
  STRUCTURE                                STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   

                STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Health statistics - Regions      I    A                   Annual   
1  Health statistics - Regions      I    A                   Annual   
2  Health statistics - Regions      I    A                   Annual   
3  Health statistics - Regions      I    A                   Annual   
4  Health statistics - Regions      I    A            

In [163]:
# Check the data structure and identify time-related columns
print("All columns in the dataset:")
print(health_df.columns.tolist())

print("\nLooking for time/date related columns:")
time_cols = [col for col in health_df.columns if any(keyword in col.lower() 
            for keyword in ['time', 'date', 'year', 'period', 'obs_time'])]
print("Time-related columns:", time_cols)

# Check the first few values of potential time columns
for col in time_cols:
    print(f"\n{col} sample values:")
    print(health_df[col].dropna().unique()[:15])

print(f"\nDataset shape: {health_df.shape}")

All columns in the dataset:
['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'TERRITORIAL_TYPE', 'Territorial typology', 'MEASURE', 'Measure', 'AGE', 'Age', 'SEX', 'Sex', 'UNIT_MEASURE', 'Unit of measure', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country', 'OBS_STATUS', 'Observation status', 'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals']

Looking for time/date related columns:
Time-related columns: ['TIME_PERIOD', 'Time period']

TIME_PERIOD sample values:
[2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023
 2024]

Time period sample values:
[]

Dataset shape: (20022, 32)


In [ ]:

vars = health_df[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
data_2022 = vars[vars['TIME_PERIOD'] == 2022]
data_2020 = vars[vars['TIME_PERIOD'] == 2020]
data_2022_us = vars[vars['COUNTRY'] == 'USA']
# Separate the two territorial levels
Health_country_data = vars[vars['Territorial level'] == 'Country']
Health_regional_data = vars[vars['Territorial level'] == 'TL2']

# Make sure TIME_PERIOD is numeric
health_df['TIME_PERIOD'] = pd.to_numeric(health_df['TIME_PERIOD'], errors='coerce')

# Sort by country and year (descending)
sorted_df = health_df.sort_values(['REF_AREA', 'TIME_PERIOD', 'Sex'], ascending=[True, False, False])

# Keep only the most recent entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='first')

circa_2024 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
Health_country_data = circa_2024[circa_2024['Territorial level'] == 'Country']
Health_regional_data = circa_2024[circa_2024['Territorial level'] == 'TL2']

In [190]:
Health_country_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_countries_complete.csv", index=False)
Health_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_regional_data.csv", index=False)

# Under5 - mortality

In [198]:
# 

BASE   = "https://sdmx.oecd.org/public/rest"
AGENCY = "OECD.CFE.EDS"
FLOW   = "DSD_REG_HEALTH@DF_HEALTH"
VER    = "2.0"

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
    "Accept": "application/json,text/csv,*/*;q=0.8",
}

# 1) Get dimension structure
probe = requests.get(
    f"{BASE}/data/{AGENCY},{FLOW},{VER}",
    params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
    headers=headers, timeout=60
)
probe.raise_for_status()
js = probe.json()

series_dims = js["structure"]["dimensions"]["series"]
dim_order = [d["id"] for d in series_dims]
print("life expectancy dataset dimensions:", dim_order)

# Build codes lookup
codes = {
    d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))}
              for v in d.get("values", [])]
    for d in series_dims
}

def pick_code(dim, prefer_ids=(), prefer_name_contains=()):
    """Pick a valid code for a dimension using id or name hints."""
    vals = codes.get(dim, [])
    # Try id preferences
    for pid in prefer_ids:
        for v in vals:
            if v["id"].upper() == pid.upper():
                return v["id"]
    # Try name contains
    for substr in prefer_name_contains:
        for v in vals:
            if substr.lower() in str(v["name"]).lower():
                return v["id"]
    # Fallback: first available (if any)
    return vals[0]["id"] if vals else ""

# 2) Build proper key with ALL 10 dimensions for education dataset
# Build proper key for health dataset
freq = pick_code("FREQ", prefer_ids=("A",))
terr_level = "TL2+CTRY"
ref_area = ""  # All regions/countries, or specify as in SDMX code
terr_type = "" # Usually empty for health
measure = "MORT_INFANT" # Life expectancy
age = "Y_GE15+Y_LT15+Y_LT1+Y0"              # All ages
sex = ""  # Or "F+M+_T" if you want all sexes
unit = "" # Usually empty for health

# Build complete key (omit education_lev and stat_op)
key_parts = [freq, terr_level, ref_area, terr_type, measure, age, sex, unit]
key = ".".join(key_parts)
print(f"Health key ({len(key_parts)} parts):", key)

# 3) Download data
url = f"{BASE}/data/{AGENCY},{FLOW},{VER}/{key}"
params = {
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels",
    "startPeriod": "2010",
    "endPeriod": "2024",
}

r = requests.get(url, params=params, headers=headers, timeout=60)

try:
    r.raise_for_status()
    health_df = pd.read_csv(io.StringIO(r.text))
    print(f"\n Health data loaded: {health_df.shape}")
    print("\nFirst few rows:")
    print(health_df.head())

    # Show what we got
    print("\nDataset contains:")
    for col in health_df.columns:
        if any(keyword in col.lower() for keyword in ['measure', 'health', 'age', 'territorial']):
            unique_vals = health_df[col].dropna().unique()
            print(f"  {col}: {unique_vals[:3]}{'...' if len(unique_vals) > 3 else ''}")
except requests.HTTPError as e:
    print(f" Still failed with: {e}")
    print("Available dimension codes:")
    for d in dim_order:
        print(f"  {d}: {[v['id'] for v in codes.get(d, [])][:5]}")

life expectancy dataset dimensions: ['FREQ', 'TERRITORIAL_LEVEL', 'REF_AREA', 'TERRITORIAL_TYPE', 'MEASURE', 'AGE', 'SEX', 'UNIT_MEASURE']
Health key (8 parts): A.TL2+CTRY...MORT_INFANT.Y_GE15+Y_LT15+Y_LT1+Y0..

 Health data loaded: (18483, 32)

First few rows:
  STRUCTURE                                STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   

                STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Health statistics - Regions      I    A                   Annual   
1  Health statistics - Regions      I    A                   Annual   
2  Health statistics - Regions      I    A                   Annual   
3  Health statistics - Regions      I    A                   Annual   
4  Health statistics - Regio

In [199]:
vars = health_df[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
data_2022 = vars[vars['TIME_PERIOD'] == 2022]
data_2020 = vars[vars['TIME_PERIOD'] == 2020]
data_2022_us = vars[vars['COUNTRY'] == 'USA']
# Separate the two territorial levels
Health_country_data = vars[vars['Territorial level'] == 'Country']
Health_regional_data = vars[vars['Territorial level'] == 'TL2']

# Make sure TIME_PERIOD is numeric
health_df['TIME_PERIOD'] = pd.to_numeric(health_df['TIME_PERIOD'], errors='coerce')

# Sort by country and year (descending)
sorted_df = health_df.sort_values(['REF_AREA', 'TIME_PERIOD', 'Sex'], ascending=[True, False, False])

# Keep only the most recent entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='first')

circa_2024 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
Health_country_data = circa_2024[circa_2024['Territorial level'] == 'Country']
Health_regional_data = circa_2024[circa_2024['Territorial level'] == 'TL2']

In [200]:
Health_country_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_countries_complete.csv", index=False)
Health_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_regional_data.csv", index=False)

## GDP

In [201]:
gdp_df = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\GDP.csv")

In [3]:
import io
import math
import requests
import pandas as pd
from typing import Dict, Iterable, List, Mapping, Optional, Tuple, Union

BASE = "https://sdmx.oecd.org/public/rest"

# ----------------------------- Core helpers -------------------------------- #

def probe_structure(
    agency: str,
    flow: str,
    ver: str,
    base_url: str = BASE,
    timeout: int = 60
) -> Tuple[List[str], Dict[str, List[Dict[str, str]]]]:
    """
    Return (dimension_order, codes_by_dimension) for a series key in an SDMX flow.
    codes_by_dimension[dim] -> list of dicts: {"id": "<CODE>", "name": "<Label>"}
    """
    r = requests.get(
        f"{base_url}/data/{agency},{flow},{ver}",
        params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
        headers={
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
            "Accept": "application/json,text/csv,*/*;q=0.8",
        },
        timeout=timeout
    )
    r.raise_for_status()
    js = r.json()

    series_dims = js["structure"]["dimensions"]["series"]
    dim_order = [d["id"] for d in series_dims]
    codes = {
        d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))}
                  for v in d.get("values", [])]
        for d in series_dims
    }
    return dim_order, codes


def _pick_code_from(
    values: List[Mapping[str, str]],
    prefer_ids: Iterable[str] = (),
    prefer_name_contains: Iterable[str] = (),
) -> str:
    """Pick a single code from a values list by id or name contains; fallback to first."""
    # try explicit ids
    prefer_ids = [p.upper() for p in prefer_ids]
    for pid in prefer_ids:
        for v in values:
            if v["id"].upper() == pid:
                return v["id"]
    # try substring on label/name
    for substr in prefer_name_contains:
        s = substr.lower()
        for v in values:
            if s in str(v.get("name", "")).lower():
                return v["id"]
    # fallback: first available
    return values[0]["id"] if values else ""


def _normalize_selector(
    dim_id: str,
    values: List[Mapping[str, str]],
    selector: Union[str, Iterable[str], Mapping[str, Iterable[str]]]
) -> str:
    """
    Return a valid SDMX code expression for this dimension:
      - str -> returned as-is
      - iterable[str] -> joined with '+'
      - dict with keys 'prefer_ids' and/or 'prefer_name_contains' -> picked by _pick_code_from
    """
    if isinstance(selector, str):
        return selector  # may be "", "TL2+CTRY", "ISCED11_5T8", etc.
    if isinstance(selector, Mapping):
        return _pick_code_from(
            values,
            selector.get("prefer_ids", ()),
            selector.get("prefer_name_contains", ()),
        )
    # assume iterable of codes (multi-select)
    try:
        codes = list(selector)
        return "+".join(codes)
    except TypeError:
        raise TypeError(f"Unsupported selector type for dimension {dim_id}: {type(selector)}")


def build_key(
    dim_order: List[str],
    codes_by_dim: Dict[str, List[Mapping[str, str]]],
    selectors: Mapping[str, Union[str, Iterable[str], Mapping[str, Iterable[str]]]],
    allow_missing: bool = True
) -> str:
    """
    Build a full SDMX key string (dot-separated) in the dataset's dimension order.
    `selectors` maps dimension -> selector (string, list of strings, or preference dict).
    If a dimension isn't specified:
      - if allow_missing=True, pick a reasonable default (first available, or empty "")
      - else raise KeyError
    """
    parts = []
    for dim in dim_order:
        if dim in selectors:
            expr = _normalize_selector(dim, codes_by_dim.get(dim, []), selectors[dim])
        else:
            if not allow_missing:
                raise KeyError(f"Selector missing for dimension: {dim}")
            # empty string often means 'all' for OECD CFE EDS in some dims (e.g., REF_AREA)
            vals = codes_by_dim.get(dim, [])
            expr = vals[0]["id"] if vals else ""
        parts.append(expr)
    return ".".join(parts)


def fetch_sdmx_csv(
    agency: str,
    flow: str,
    ver: str,
    selectors: Mapping[str, Union[str, Iterable[str], Mapping[str, Iterable[str]]]],
    *,
    base_url: str = BASE,
    start: Optional[str] = None,
    end: Optional[str] = None,
    dimension_at_obs: str = "AllDimensions",
    timeout: int = 60,
    verbose: bool = False
) -> Tuple[pd.DataFrame, Dict[str, List[Mapping[str, str]]], List[str], str]:
    """
    Probe structure, build key from `selectors`, and return (df, codes_by_dim, dim_order, key).
    `selectors` accepts:
      - exact code string: "A", "TL2+CTRY", "", "ISCED11_5T8"
      - list of codes: ["F","M","_T"] -> "F+M+_T"
      - preference dict: {"prefer_ids": ("A",), "prefer_name_contains": ("total",)}
    """
    dim_order, codes = probe_structure(agency, flow, ver, base_url=base_url, timeout=timeout)
    key = build_key(dim_order, codes, selectors)

    params = {"format": "csvfilewithlabels", "dimensionAtObservation": dimension_at_obs}
    if start:
        params["startPeriod"] = start
    if end:
        params["endPeriod"] = end

    url = f"{base_url}/data/{agency},{flow},{ver}/{key}"
    r = requests.get(
        url,
        params=params,
        headers={
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
            "Accept": "application/json,text/csv,*/*;q=0.8",
        },
        timeout=timeout
    )
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        if verbose:
            print("HTTP error:", e)
            print("Key:", key)
            print("Dimensions:", dim_order)
            for d in dim_order:
                print(f"{d}: {[v['id'] for v in codes.get(d, [])][:8]} ...")
        raise

    df = pd.read_csv(io.StringIO(r.text))
    if verbose:
        print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} cols\nKey: {key}")
    return df, codes, dim_order, key

# ----------------------------- Ready-made examples -------------------------- #

def load_education_attainment_tertiary(
    start="2010", end="2024", verbose=False
) -> pd.DataFrame:
    """
    OECD CFE EDS — Education attainment (tertiary, ages 25–64), TL2 + countries, annual, % of pop.
    """
    agency = "OECD.CFE.EDS"
    flow = "DSD_REG_EDU@DF_ATTAIN"
    ver = "2.0"

    selectors = {
        # Annual
        "FREQ": {"prefer_ids": ("A",)},
        # Both TL2 regions and countries
        "TERRITORIAL_LEVEL": "TL2+CTRY",
        # All areas (leave empty in this flow to mean 'all')
        "REF_AREA": "",
        # Not applicable / default territorial type
        "TERRITORIAL_TYPE": {"prefer_ids": ("_Z",)},
        # Education measure: education attainment share
        "MEASURE": {"prefer_ids": ("NEAC_SHARE_EA",)},
        # Age 25–64
        "AGE": {"prefer_ids": ("Y25T64",)},
        # Sex total
        "SEX": {"prefer_ids": ("_T", "T"), "prefer_name_contains": ("total",)},
        # Tertiary (ISCED 5–8)
        "EDUCATION_LEV": {"prefer_ids": ("ISCED11_5T8",)},
        # Mean
        "STATISTICAL_OPERATION": {"prefer_ids": ("MEAN",)},
        # Percentage in population by sex-age
        "UNIT_MEASURE": {"prefer_ids": ("PT_POP_SEX_AGE",)},
    }

    df, *_ = fetch_sdmx_csv(
        agency, flow, ver, selectors, start=start, end=end, verbose=verbose
    )
    return df

def load_enrolment_rate(
    *,
    ref_areas,                         # list like ["AUT","AUS","AU1",...]
    age: str = "Y15T19",
    sexes=("F","M","_T"),
    territorial_level: str = "CTRY+TL2",
    start="2013",
    end=None,
    verbose=False
):
    """
    OECD CFE EDS — Education Enrolment Rate (DF_EDU).
    Example measure: ENRL_RATE for ages 15–19, by TL2 + Country.
    """
    agency = "OECD.CFE.EDS"
    flow   = "DSD_REG_EDU@DF_EDU"
    ver    = "2.0"

    selectors = {
        "FREQ": {"prefer_ids": ("A",)},              # Annual
        "TERRITORIAL_LEVEL": territorial_level,      # "CTRY+TL2"
        "REF_AREA": ref_areas,                       # list -> "AUT+AUS+AU1+..."
        "TERRITORIAL_TYPE": "",                      # empty is common here
        "MEASURE": "ENRL_RATE",
        "AGE": age,                                  # "Y15T19"
        "SEX": list(sexes),                          # e.g. ("F","M","_T")
        "EDUCATION_LEV": "_T",                       # total level
        # Many DF_EDU series either omit these or default; let the picker choose.
        "STATISTICAL_OPERATION": {"prefer_ids": ("MEAN",)},
        "UNIT_MEASURE": {"prefer_ids": ("PT_POP_SEX_AGE","PT","RATE")},
    }

    df, *_ = fetch_sdmx_csv(
        agency, flow, ver, selectors, start=start, end=end, verbose=verbose
    )
    return df

def load_health(
    measure_code: str,
    *,
    ages: Union[str, Iterable[str]] = "Y0",
    sexes: Union[str, Iterable[str]] = "",
    start="2010",
    end="2024",
    verbose=False
) -> pd.DataFrame:
    """
    OECD CFE EDS — Health (life expectancy, infant mortality, etc.), TL2 + countries, annual.
    - measure_code examples: "LFEXP" (life expectancy), "MORT_INFANT" (infant mortality)
    - ages: string or list like ["Y_GE15","Y_LT15","Y_LT1","Y0"] -> becomes "Y_GE15+Y_LT15+Y_LT1+Y0"
    - sexes: "", "_T", or list like ["F","M","_T"]
    """
    agency = "OECD.CFE.EDS"
    flow = "DSD_REG_HEALTH@DF_HEALTH"
    ver = "2.0"

    selectors = {
        "FREQ": {"prefer_ids": ("A",)},
        "TERRITORIAL_LEVEL": "TL2+CTRY",
        "REF_AREA": "",
        "TERRITORIAL_TYPE": "",
        "MEASURE": measure_code,                 # e.g., "LFEXP" or "MORT_INFANT"
        "AGE": ages,                             # e.g., "Y0" or ["Y_GE15","Y_LT15","Y_LT1","Y0"]
        "SEX": sexes,                            # e.g., "" or "_T" or ["F","M","_T"]
        "UNIT_MEASURE": "",                      # usually blank for HEALTH
    }

    df, *_ = fetch_sdmx_csv(
        agency, flow, ver, selectors, start=start, end=end, verbose=verbose
    )
    return df

# -------------------------------- Usage examples --------------------------- #



In [ ]:
if __name__ == "__main__":
    # 1) Education: tertiary attainment, 25–64
    edu = load_education_attainment_tertiary(start="2010", end="2024", verbose=True)
    print("Education sample:")
    print(edu.head())

    # 2) Health: life expectancy at birth (LFEXP), total sex, age Y0
    lifeexp = load_health("LFEXP", ages="Y0", sexes="_T", start="2010", end="2024", verbose=True)
    print("Life expectancy sample:")
    print(lifeexp.head())

    # 3) Health: infant mortality, multiple age buckets combined
    mort_inf = load_health("MORT_INFANT", ages=["Y_GE15", "Y_LT15", "Y_LT1", "Y0"], sexes="_T",
                           start="2010", end="2024", verbose=True)
    print("Infant mortality sample:")
    print(mort_inf.head())

    # 4) Education: enrolment rate for ages 15–19,
    enrol_15_19 = load_enrolment_rate(
    ref_areas="",
    age="Y15T19",
    sexes=("F","M","_T"),
    start="2010",
    end=2024,         # or "2024"
    verbose=True)
    print("Enrolment rate sample:")
    print(enrol_15_19.head())




Loaded 6,771 rows × 36 cols
Key: A.TL2+CTRY.._Z.NEAC_SHARE_EA.Y25T64._T.ISCED11_5T8.MEAN.PT_POP_SEX_AGE
Education sample:
  STRUCTURE                             STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   

                     STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Educational attainment - Regions      I    A                   Annual   
1  Educational attainment - Regions      I    A                   Annual   
2  Educational attainment - Regions      I    A                   Annual   
3  Educational attainment - Regions      I    A                   Annual   
4  Educational attainment - Regions      I    A                   Annual   

  TERRITORIAL_LEVEL Territorial level REF_AREA Reference area  ... OBS_VALUE  \
0   

In [84]:
vars = enrol_15_19[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
data_2022 = vars[vars['TIME_PERIOD'] == 2022]
data_2020 = vars[vars['TIME_PERIOD'] == 2020]
data_2022_us = vars[vars['COUNTRY'] == 'USA']
# Separate the two territorial levels
enrollment_country_data = vars[vars['Territorial level'] == 'Country']
enrollment_regional_data = vars[vars['Territorial level'] == 'TL2']

# Make sure TIME_PERIOD is numeric
enrollment_country_data['TIME_PERIOD'] = pd.to_numeric(enrollment_country_data['TIME_PERIOD'], errors='coerce')

# Sort by country and year (descending)
sorted_df = enrol_15_19.sort_values(['REF_AREA', 'TIME_PERIOD', 'Sex'], ascending=[True, False, False])

# Keep only the most recent entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='first')

circa_2024 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
enrollment_country_data = circa_2024[circa_2024['Territorial level'] == 'Country']
enrollment_regional_data = circa_2024[circa_2024['Territorial level'] == 'TL2']


C:\Users\lopez\AppData\Local\Temp\ipykernel_49284\362401194.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enrollment_country_data['TIME_PERIOD'] = pd.to_numeric(enrollment_country_data['TIME_PERIOD'], errors='coerce')


In [85]:
enrollment_country_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_countries_complete.csv", index=False)
enrollment_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_regional_data.csv", index=False)

In [20]:
regional_with_gap = {}

for name, df in indicators.items():
    # 1) TL2-only
    reg = df[df["Territorial level"] == "TL2"].copy()

    # 2) Ensure a 'Country' column to group by
    if "Country" not in reg.columns:
        # Try common alternatives from OECD CSVs:
        for candidate in ["Reference country", "REF_AREA_PARENT", "REF_AREA_COUNTRY"]:
            if candidate in reg.columns:
                reg["Country"] = reg[candidate]
                break
        else:
            # Fallback: derive from REF_AREA (adjust to your coding scheme)
            # Many TL2 codes start with the ISO3 country, or at least 3-letter country.
            reg["Country"] = reg["REF_AREA"].str[:3]

    # 3) Gap per (year, country): max - min across TL2 regions
    gap_by_year_country = (
        reg.groupby(["TIME_PERIOD", "Country"], as_index=False)["OBS_VALUE"]
           .agg(gap_obs_value=lambda x: x.max() - x.min())
    )

    # 4) Attach the gap to every regional row
    reg = reg.merge(gap_by_year_country, on=["TIME_PERIOD", "Country"], how="left")

    # 5) Store result
    regional_with_gap[name] = reg

In [21]:
regional_with_gap["edu"]

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,FREQ,Frequency of observation,TERRITORIAL_LEVEL,Territorial level,REF_AREA,Reference area,...,Observation value,COUNTRY,Country,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals,gap_obs_value
0,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,US51,Virginia,...,NaN,USA,United States,A,Normal value,0,Units,1,One,30.6
1,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,US32,Nevada,...,NaN,USA,United States,A,Normal value,0,Units,1,One,30.6
2,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,US26,Michigan,...,NaN,USA,United States,A,Normal value,0,Units,1,One,30.6
3,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,US38,North Dakota,...,NaN,USA,United States,A,Normal value,0,Units,1,One,30.6
4,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,US41,Oregon,...,NaN,USA,United States,A,Normal value,0,Units,1,One,30.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6097,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,HU32,Northern Great Plain,...,NaN,HUN,Hungary,B,Time series break,0,Units,1,One,36.8
6098,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,HR02,Pannonian Croatia,...,NaN,HRV,Croatia,B,Time series break,0,Units,1,One,26.1
6099,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,FRI,Nouvelle-Aquitaine,...,NaN,FRA,France,B,Time series break,0,Units,1,One,36.2
6100,DATAFLOW,OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0),Educational attainment - Regions,I,A,Annual,TL2,TL2,ITH3,Veneto,...,NaN,ITA,Italy,B,Time series break,0,Units,1,One,11.3


In [33]:
all_gaps = {}

for name, reg in regional_with_gap.items():
    # Group by country + year
    gap_table = (
        reg.groupby(["Country", "TIME_PERIOD"])
        .agg(
            gap_obs_value=("gap_obs_value", "first"),  # same per country-year
            n_regions=("REF_AREA", "nunique")          # number of TL2 regions in that year
        )
        .reset_index()
        .sort_values(["Country", "TIME_PERIOD"])
    )

    # Find countries with >5 unique TL2 regions across the whole dataset
    region_counts = (
        reg.groupby("Country")["REF_AREA"]
        .nunique()
    )
    countries_with_5plus = region_counts[region_counts > 5].index

    # Filter gap_table to only those countries
    gap_table = gap_table[gap_table["Country"].isin(countries_with_5plus)]

    all_gaps[name] = gap_table

# Example: see for education
print("Education gaps (countries with >5 TL2 regions):")
print(all_gaps["edu"].head())


Education gaps (countries with >5 TL2 regions):
     Country  TIME_PERIOD  gap_obs_value  n_regions
0  Australia         2016           28.9          8
1  Australia         2017           26.5          8
2  Australia         2018           31.5          8
3  Australia         2019           25.2          8
4  Australia         2020           24.0          8


In [91]:
selected_countries = [
    "United States", "Czechia",  "Canada", "Spain",
    "France", "United Kingdom", "Colombia", "Romania",  
    "Greece", "Bulgaria", "Mexico", "Chile"
    
]

filtered_edu = all_gaps["edu"][all_gaps["edu"]["Country"].isin(selected_countries)]



In [92]:
df = filtered_edu  # columns: Country, TIME_PERIOD, gap_obs_value, n_regions

# add a per-country sort key (mean gap)
base = (
    alt.Chart(df)
    .transform_joinaggregate(sort_key='mean(gap_obs_value)', groupby=['Country'])
)

line = (
    base.mark_line(color="#9C0505")
    .encode(
        x=alt.X('TIME_PERIOD:O', title='Year', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('gap_obs_value:Q', title='Gap (pp, max–min of TL2)'),
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions']
    )
)

points = (
    base.mark_point(color='red', filled=True, size=50)
    .encode(
        x='TIME_PERIOD:O',
        y='gap_obs_value:Q'
    )
)

panel = (line + points).properties(width=160, height=120)

panel = (
    base.mark_line(color='#9C0505', point=True)
    .encode(
        x=alt.X('TIME_PERIOD:O', title='Year', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('gap_obs_value:Q', title='Gap (pp, max–min of TL2)'),
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions']
    )
    .properties(width=160, height=120)
)

chart = panel.facet(
    facet=alt.Facet(
        'Country:N',
        sort=alt.SortField(field='sort_key', order='descending'),  # << fix
        title=None
    ),
    columns=6
).resolve_scale(y='shared').properties(
    title='Education Attainment Gap (Tertiary, 25–64) — Selected Countries (2010–2023)'
).configure_axis(grid=True)

chart


alt.FacetChart(...)

In [94]:
# ...existing code...
df = filtered_edu  # columns: Country, TIME_PERIOD, gap_obs_value, n_regions

# compute first / middle / last year values (fall back to whatever exists)
_years = sorted(df['TIME_PERIOD'].dropna().unique())
if len(_years) >= 3:
    axis_years = [int(_years[0]), int(_years[len(_years)//2]), int(_years[-1])]
else:
    axis_years = [int(y) for y in _years]

x_axis = alt.X('TIME_PERIOD:O', title='Year', axis=alt.Axis(values=axis_years, labelAngle=0))

# add a per-country sort key (mean gap)
base = (
    alt.Chart(df)
    .transform_joinaggregate(sort_key='mean(gap_obs_value)', groupby=['Country'])
)

line = (
    base.mark_line()
    .encode(
        x=x_axis,
        y=alt.Y('gap_obs_value:Q', title='Gap (pp, max–min of TL2)'),
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions'],
        color=alt.value("#9C0505")
    )
)

points = (
    base.mark_point(filled=True, size=50)
    .encode(
        x=x_axis,
        y='gap_obs_value:Q',
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions'],
        color=alt.value("#9C0505")
    )
)

panel = (line + points).properties(width=160, height=120)

chart = panel.facet(
    facet=alt.Facet(
        'Country:N',
        sort=alt.EncodingSortField(field='sort_key', order='descending'),
        title=None
    ),
    columns=6
).resolve_scale(y='shared').properties(
    title='Education Attainment Gap (Tertiary, 25–64) — Selected Countries (2010–2023)'
).configure_axis(grid=True)

chart

chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\trend_education_attainment_gap_selected_countries.png")